In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.metrics import roc_auc_score,balanced_accuracy_score, recall_score

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


### Load the embeddings for the C.S.sylv sulcal region

In [505]:
#ukb_embeddings = pd.read_csv('/neurospin/dico/data/deep_folding/current/models/Champollion_V0_trained_on_UKB40/SC-sylv_right/11-36-10_85_0/ukb40_random_epoch80_embeddings/full_embeddings.csv', index_col=0)
#ukb_embeddings = pd.read_csv('/neurospin/dico/data/deep_folding/current/models/Champollion_V0/SC-sylv_right/11-43-38_3/ukb40_random_epoch100_embeddings/full_embeddings.csv', index_col=0)
ukb_embeddings = pd.read_csv('/neurospin/dico/data/deep_folding/current/models/Champollion_V0/SC-sylv_right/13-19-08_28/ukb40_random_epoch80_embeddings/full_embeddings.csv', index_col=0)
print(ukb_embeddings.shape)
ukb_embeddings.head()

(42433, 256)


,dim1,dim2,dim3,dim4,dim5,dim6,dim7,dim8,dim9,dim10,...,dim247,dim248,dim249,dim250,dim251,dim252,dim253,dim254,dim255,dim256
ID,,,,,,,,,,,,,,,,,,,,,
sub-1000021,-18.050947,-1.715077,-12.134333,-7.756078,-34.884075,52.562733,-55.73811,9.456552,-32.501210,-5.591976,...,-29.091085,-88.040250,63.468884,65.100220,4.362944,-23.781483,-1.971074,-35.363983,0.652892,-15.119811
sub-1000325,34.017770,3.306584,-1.062342,16.128439,-11.552667,56.742270,-63.84294,-9.990108,-32.844690,-6.814365,...,-17.474247,-5.164792,130.832440,25.999964,28.174246,-5.317329,12.176266,12.650237,-2.148138,-20.165537
sub-1000458,-40.606728,-60.641052,-68.245430,5.354382,-0.104062,-49.856945,-104.72249,-8.398323,-11.103157,28.699837,...,-43.523790,-66.923620,38.453907,52.463734,5.161959,10.050982,-0.961989,-18.190912,-83.235630,10.441000
sub-1000575,29.307630,-28.164337,-68.444870,0.264590,-12.987295,26.187790,-80.72651,13.687674,36.764324,-1.856947,...,-34.156240,-57.731100,44.394917,22.953100,11.024565,7.410131,12.635066,-33.951286,-54.387440,13.403832
sub-1000606,-48.289925,-70.598090,15.068221,-70.852080,44.615005,16.564405,-168.66982,6.039364,34.128735,1.094133,...,23.377523,-95.845436,20.168795,109.883480,-52.428160,12.407044,-19.002588,-61.777813,-30.722471,-33.493202


### Reduce dimension (hope to remove the noise) with a PCA

In [506]:
n_components=40

pca = PCA(n_components=n_components)
pca.fit(ukb_embeddings)
print(pca.explained_variance_ratio_)
(np.cumsum(pca.explained_variance_ratio_) < 0.99).sum()

[1.80767982e-01 1.30870328e-01 1.20435526e-01 1.13013054e-01
 1.04927850e-01 8.61114994e-02 7.56570315e-02 6.33808940e-02
 3.66405718e-02 2.43834487e-02 1.31787766e-02 1.12454856e-02
 1.02233831e-02 7.60673772e-03 5.04616026e-03 3.20213495e-03
 2.77098143e-03 1.90791256e-03 1.59604904e-03 1.25505022e-03
 9.52785493e-04 6.92517428e-04 5.86084217e-04 4.50962100e-04
 3.46854135e-04 2.82669135e-04 2.56894576e-04 2.34469116e-04
 1.66603781e-04 1.54011133e-04 1.26263042e-04 1.19290882e-04
 1.05772616e-04 1.02354039e-04 9.33541042e-05 8.30737230e-05
 7.71875896e-05 5.60097206e-05 5.53050114e-05 5.12705358e-05]


17

In [507]:
ukb_pca_bdd = pca.transform(ukb_embeddings)

In [508]:
#scaler = StandardScaler()
#scaler.fit(ukb_embeddings)
#ukb_scl_bdd = scaler.transform(ukb_embeddings)
#ukb_scl_bdd

In [567]:
interrupted = [
'sub-1310920',
'sub-1376904',
'sub-2863742',
'sub-3694216',
'sub-1037052',
'sub-3250551',
'sub-5401486',
'sub-1499791', # good
'sub-1911266',
'sub-4217758',
'sub-2693192',
'sub-1633860',
'sub-5222070',
'sub-3292254',
'sub-1613821',
'sub-2771619',
'sub-3159828',
'sub-4632483',
'sub-5936108',
'sub-3794487',
'sub-1420697', # not sure
'sub-1111996', # not sure
'sub-1425827', # not sure
'sub-2846621', # good
'sub-2004479',
'sub-3891499',
'sub-5236788',
'sub-3061407', # very good
'sub-5693167',
'sub-2155264', # very good
'sub-2444973', # very good
'sub-5245412', # good
'sub-5574911', # very good
'sub-2852894', # very good
'sub-1106033', # very good
'sub-5984646', # very good
'sub-5739487', # very good
'sub-3492298', # good
'sub-5712569', # not sure
'sub-2200121', # not sure
'sub-5638090', # good
'sub-4496792', # good
'sub-5129881', # good
'sub-1775041', # good
'sub-1094593', # good
'sub-1358401', # good
'sub-4354208', # very good
'sub-1428452', # good
'sub-5731125',
'sub-4995189', # very good
'sub-1499791',
'sub-2762943', # very good
'sub-3386408', # not sure
'sub-5665554', # not sure
'sub-1130686', # good
'sub-2484762', # good
'sub-5186095', # good
'sub-5569356',
'sub-4762603', # not sure
'sub-3572724', # good
'sub-2573795', # good
'sub-5315648',
'sub-2731992',
'sub-4949978',
'sub-2776534',
'sub-2298245',
'sub-2570335',
'sub-3258249', # not sure
'sub-1748817', # not sure
'sub-4203366', # very good
'sub-4184635', # very good
'sub-1053493', # note sure
'sub-3947538', # not sure
'sub-2741631', # not sure
'sub-3492301',
'sub-4447456',
'sub-2373286', # not sure
'sub-4732282',
'sub-3293670',
'sub-2149638', # good
'sub-4625643', # not sure
'sub-4328267',
'sub-2589361',
'sub-4232003',
'sub-5456948',
'sub-4420000', # not sure
'sub-1553423', # not sure
'sub-1405899', # not sure
'sub-2550690', # not sure
'sub-2986522', # not sure
'sub-1698233', # not sure
'sub-4603077', # not sure
'sub-3428215',
'sub-1935008',
'sub-4589882',
'sub-2323818',
'sub-2230154',
'sub-1675253',
'sub-4875056',
'sub-4059279',
'sub-4067363',
'sub-1322441',
'sub-1417407',
'sub-3733675',
'sub-5531350',
'sub-4067363',
'sub-1369171',
'sub-1807186',
'sub-1267836',
'sub-3758439',
'sub-4652131',
'sub-1052521',
'sub-5949398',
'sub-3672666',
'sub-4754998',
'sub-3791185',
'sub-4587270',
'sub-4599903',
'sub-5617588',
'sub-1428212',
'sub-3911620',
'sub-4152006',
'sub-1864685',
'sub-5366951',
'sub-3679537',
'sub-5209589',
'sub-4211996',
'sub-3913796',
'sub-5777436',
'sub-4340260',
'sub-1132414',
'sub-5428293',
'sub-5406975',
'sub-4286421',
'sub-3253763',
'sub-1154509',
'sub-3500106',
'sub-4779638',
'sub-1107908',
'sub-4186067',
'sub-4916661',
'sub-3489143',
'sub-4212266',
'sub-2487105',
'sub-1569878',
'sub-5946014',
'sub-4561628',
'sub-5993265',
'sub-3494489',
'sub-1421577',
'sub-2916349',
'sub-4935525',
'sub-3694967',
'sub-3450882',
'sub-1233271',
'sub-2519386',
'sub-2512500',
'sub-1950476',
'sub-2662418',
'sub-3719519',
'sub-3450499',
'sub-2310242',
'sub-3842960',
'sub-5492484',
'sub-4862708',
'sub-4833497',
]

not_interrupted = [
'sub-1103646',
 'sub-1167379',
 'sub-1190643',
 'sub-1273718',
 'sub-1286007',
 'sub-1298876',
 'sub-1352284',
 'sub-1398736',
 'sub-1422413',
 'sub-1465129',
 'sub-1597706',
 'sub-1701563',
 'sub-1734788',
 'sub-1979982',
 'sub-1996092',
 'sub-2005939',
 'sub-2036033',
 'sub-2097565',
 'sub-2118136',
 'sub-2141551',
 'sub-2193253',
 'sub-2207793',
 'sub-2228486',
 'sub-2284024',
 'sub-2337820',
 'sub-2349203',
 'sub-2389411',
 'sub-2420937',
 'sub-2427515',
 'sub-2538754',
 'sub-2583027',
 'sub-2592717',
 'sub-2733674',
 'sub-2741815',
 'sub-2792782',
 'sub-2802489',
 'sub-2814161',
 'sub-2816262',
 'sub-2833426',
 'sub-2834970',
 'sub-2837393',
 'sub-2889389',
 'sub-2946274',
 'sub-2957401',
 'sub-2968297',
 'sub-2970418',
 'sub-3008660',
 'sub-3009279',
 'sub-3013938',
 'sub-3227039',
 'sub-3234836',
 'sub-3264612',
 'sub-3333294',
 'sub-3334219',
 'sub-3379262',
 'sub-3388080',
 'sub-3388306',
 'sub-3401499',
 'sub-3453064',
 'sub-3525594',
 'sub-3529189',
 'sub-3541105',
 'sub-3603191',
 'sub-3627711',
 'sub-3670173',
 'sub-3693543',
 'sub-3721299',
 'sub-3722413',
 'sub-3765466',
 'sub-3936967',
 'sub-3992259',
 'sub-3994474',
 'sub-4016129',
 'sub-4027732',
 'sub-4116944',
 'sub-4411765',
 'sub-4420611',
 'sub-4428393',
 'sub-4491384',
 'sub-4519441',
 'sub-4520944',
 'sub-4536778',
 'sub-4727825',
 'sub-4741296',
 'sub-4755899',
 'sub-4787289',
 'sub-4791977',
 'sub-4805119',
 'sub-4805237',
 'sub-4834994',
 'sub-4868991',
 'sub-5027399',
 'sub-5054716',
 'sub-5082433',
 'sub-5117110',
 'sub-5123219',
 'sub-5147403',
 'sub-5217534',
 'sub-5237880',
 'sub-5292898',
 'sub-5293703',
 'sub-5319071',
 'sub-5430535',
 'sub-5437419',
 'sub-5486726',
 'sub-5561142',
 'sub-5578922',
 'sub-5581707',
 'sub-5605784',
 'sub-5643778',
 'sub-5649675',
 'sub-5686761',
 'sub-5723111',
 'sub-5729132',
 'sub-5749108',
 'sub-5754849',
 'sub-5836983',
 'sub-5864979',
 'sub-5910947',
 'sub-5966409',
 'sub-5998652',
 'sub-5357627',
 'sub-2204575',
 'sub-2839753',
 'sub-4281714',
 'sub-1649070',
 'sub-5335727',
 'sub-5782466',
 'sub-4520082',
 'sub-1004170',
 'sub-4158073', 
 'sub-5684893',
 'sub-4359496',
 'sub-2040983',
 'sub-5575777',
 'sub-1116938',
 'sub-4189639',
 'sub-4507392',
 'sub-5085553',
 'sub-2077194',
 'sub-5457081',
 'sub-3692612',
 'sub-2379487',
 'sub-5137278',
 'sub-4831688',
 'sub-3976041',
 'sub-4057189',
 'sub-4202490',
 'sub-4844615',
 'sub-4747425',
 'sub-1008582',
 'sub-4039492',
 'sub-2969851', 
 'sub-2830945', 
 'sub-1711798', 
 'sub-5120758',
 'sub-4949491', 
 'sub-4772821', 
 'sub-4450785', 
 'sub-1110891',
 'sub-2920350',
 'sub-5217882',
 'sub-5747063',
 'sub-5040811',
 'sub-3143577',
 'sub-4571621',
 'sub-1779953',
 'sub-2112770',
 'sub-2327257',
 'sub-2406636',
 'sub-3745349',
 'sub-2968941',
 'sub-2611694',
 'sub-2676227',
 'sub-1164936',
 'sub-4794448',
 'sub-5711287',
 'sub-1502163',
 'sub-1546206',
 'sub-4710850',
 'sub-3735109',
 'sub-2800235',
 'sub-5115775',
 'sub-3746225',
 'sub-2425148',
 'sub-1416140',
 'sub-5907102',
 'sub-5989851',
 'sub-3974444',
 'sub-2212279',
 'sub-4899126',
 'sub-4518664',
 'sub-5828387',
 'sub-1385217',
 'sub-2442019',
 'sub-1273391',
 'sub-2302082', 
 'sub-2814161',
 'sub-3785229',
 'sub-4945539',
] 

In [568]:
"""
Problem with:
'sub-3716267'
"""

"\nProblem with:\n'sub-3716267'\n"

In [569]:
X = ukb_embeddings.loc[interrupted + not_interrupted]
y = [1 for i in range(len(interrupted))] + [0 for i in range(len(not_interrupted))]
X_pca = pca.transform(X)
len(interrupted), len(not_interrupted)

(166, 200)

In [536]:
X_train_pca, X_test_pca, y_train, y_test = train_test_split(X_pca, y, test_size=0.33, random_state=42)

#### linear SVC model

In [537]:
model = SVC(kernel='linear', probability=True,
            random_state=42,
            C=0.0001, class_weight='balanced', decision_function_shape='ovr')

In [538]:
model.fit(X_train_pca, y_train)

print('Recall:', recall_score(y_test, model.predict(X_test_pca)), '\n')
print('ROC:', roc_auc_score(y_test ,model.predict_proba(X_test_pca)[:,1]), '\n')
print('Balanced accuracy:',balanced_accuracy_score(y_test, model.predict(X_test_pca)), '\n')
model.fit(X_pca, y)

Recall: 0.7586206896551724 

ROC: 0.8459770114942529 

Balanced accuracy: 0.7543103448275862 



SVC(C=0.0001, class_weight='balanced', kernel='linear', probability=True,
    random_state=42)

In [517]:
prediction = pd.DataFrame({"IID" : list(ukb_embeddings.index),
              "Pred" : model.predict_proba(ukb_pca_bdd)[:,1]})
prediction

,IID,Pred
0,sub-1000021,0.067540
1,sub-1000325,0.043436
2,sub-1000458,0.114134
3,sub-1000575,0.057665
4,sub-1000606,0.117313
...,...,...
42428,sub-6023847,0.147329
42429,sub-6024038,0.188992
42430,sub-6024150,0.062546
42431,sub-6024379,0.064858


In [518]:
print('Maximum probability of prediction among the interrupted C.S. :',prediction[prediction["IID"].isin(interrupted)].Pred.max(), '\n')
print('Mean probability of prediction among the interrupted C.S. :',prediction[prediction["IID"].isin(interrupted)].Pred.mean(), '\n')
prediction[prediction['IID']=='sub-2036033']

Maximum probability of prediction among the interrupted C.S. : 0.9701525749293103 

Mean probability of prediction among the interrupted C.S. : 0.6723715378052666 



,IID,Pred
8695,sub-2036033,0.190922


In [519]:
((prediction[~(prediction["IID"].isin(interrupted))]).sort_values(by="Pred")[-5:].IID).to_list()

['sub-3538950', 'sub-4686469', 'sub-5646393', 'sub-1266896', 'sub-2040983']

#### Second approach: Euclidian distance in the reduced latent space

In [520]:
from scipy.spatial import distance

In [521]:
list_dist = [distance.euclidean(pca.transform(ukb_embeddings.loc['sub-3791185'].to_numpy().reshape(1,-1)), ukb_pca_bdd[i]) for i in range(len(ukb_pca_bdd))]
df_dist = pd.DataFrame({"IID":list(ukb_embeddings.index), "Dist":list_dist})

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
/usr

In [522]:
sample_dist = ((df_dist[~(df_dist["IID"].isin(interrupted))]).sort_values(by='Dist').iloc[22000:22025].IID).to_list()

### Visualization with Anatomist

In [523]:
import anatomist.api as ana
from soma.qt_gui.qtThread import QtThreadCall
from soma.qt_gui.qt_backend import Qt

a = ana.Anatomist()

from soma import aims

In [453]:
dataset = 'UkBioBank40'
region = "S.C.-sylv."
side = "R"

mm_skeleton_path = f'/neurospin/dico/data/deep_folding/current/datasets/{dataset}/crops/2mm/{region}/mask/{side}crops'

In [564]:
sample = ((prediction[~(prediction["IID"].isin(interrupted))]).sort_values(by="Pred", ascending=False)[150:175].IID).to_list()

In [565]:
volume=True
volume_files = []

for subject_id in sample:
    volume_path = f"{mm_skeleton_path}/{subject_id}_cropped_skeleton.nii.gz"
    
    if volume:
        if os.path.isfile(volume_path):
            vol = aims.read(volume_path)
            volume_files.append(vol)
        else:
            print(f"{volume_path} is not a correct path, or the .nii.gz doesn't exist")

block = a.createWindowsBlock(5) # 10 columns
dic_windows = {}

if volume:
    for i, vol in enumerate(volume_files):
        dic_windows[f'a_vol{i}'] = a.toAObject(vol)
        #dic_windows[f'a_vol{i}'].setPalette(absoluteMode=True)
        dic_windows[f'rvol{i}'] = a.fusionObjects(objects=[dic_windows[f'a_vol{i}']], method='VolumeRenderingFusionMethod')
        dic_windows[f'rvol{i}'].releaseAppRef()
        dic_windows[f'wvr{i}'] = a.createWindow('3D', block=block) #geometry=[100+400*(i%3), 100+440*(i//3), 400, 400])
        dic_windows[f'wvr{i}'].addObjects(dic_windows[f'rvol{i}'])

no position could be read at 242, 110


In [566]:
sample[8]

'sub-4945539'

In [1077]:
sample_dist[1]

['sub-5293703', 'sub-5319071', 'sub-3525594', 'sub-5561142']